# KELT-20b TESS exploration

This notebook is for exploratory TESS photometry work on KELT-20b:
- fetch SPOC 2-minute light curves
- stitch and flatten the sectors
- inspect the phase-folded transit
- inspect sector-level / epoch-level behavior
- optionally build and sample a PyMC + exoplanet transit model

It is not part of the retrieval pipeline itself. The retrieval currently uses derived scalar photometric constraints, not raw TESS light curves.



In [ ]:
import warnings

import lightkurve as lk
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning, module="pytensor")

TARGET = "KELT-20"
MISSION = "TESS"
AUTHOR = "SPOC"
EXPTIME_S = 120

PERIOD_D = 3.4741039
T0_BTJD = 1698.210775
TRANSIT_DURATION_D = 3.54 / 24.0
PLOT_WINDOW_D = 0.15

search = lk.search_lightcurve(TARGET, mission=MISSION, author=AUTHOR, exptime=EXPTIME_S)
display(search)

lc_collection = search.download_all()
print(f"Downloaded {len(lc_collection)} light curves")



In [ ]:
stitched_lc = lc_collection.stitch().remove_nans().normalize()
transit_mask = stitched_lc.create_transit_mask(
    period=PERIOD_D,
    transit_time=T0_BTJD,
    duration=TRANSIT_DURATION_D,
)
flat_lc = stitched_lc.flatten(window_length=901, mask=transit_mask)
folded_lc = flat_lc.fold(period=PERIOD_D, epoch_time=T0_BTJD)
binned_folded_lc = folded_lc.bin(time_bin_size=5.0 / (24.0 * 60.0))

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)

stitched_lc.scatter(ax=axes[0], color="0.35", alpha=0.35, s=4)
axes[0].set_title("KELT-20 stitched TESS light curve")
axes[0].set_ylabel("Normalized flux")

folded_lc.scatter(ax=axes[1], color="0.6", alpha=0.18, s=6, label="Unbinned")
binned_folded_lc.scatter(ax=axes[1], color="tab:blue", s=18, label="5 min bins")
axes[1].set_xlim(-PLOT_WINDOW_D, PLOT_WINDOW_D)
axes[1].set_title("Phase-folded transit")
axes[1].set_xlabel("Time from mid-transit [days]")
axes[1].set_ylabel("Normalized flux")
axes[1].legend()

plt.tight_layout()



In [ ]:
sector_lc = lc_collection[0].remove_nans().normalize()
sector_transit_mask = sector_lc.create_transit_mask(
    period=PERIOD_D,
    transit_time=T0_BTJD,
    duration=TRANSIT_DURATION_D,
)
sector_flat_lc = sector_lc.flatten(window_length=901, mask=sector_transit_mask)
sector_folded_lc = sector_flat_lc.fold(period=PERIOD_D, epoch_time=T0_BTJD)

fig, ax = plt.subplots(figsize=(10, 5))
sector_folded_lc.plot_river(ax=ax)
ax.set_aspect("auto")
ax.set_title("KELT-20b river plot for the first downloaded TESS sector")
plt.tight_layout()



In [ ]:
time_btjd = flat_lc.time.value
phase_days = ((time_btjd - T0_BTJD + 0.5 * PERIOD_D) % PERIOD_D) - 0.5 * PERIOD_D
epoch_number = np.round((time_btjd - T0_BTJD) / PERIOD_D).astype(int)

fig, ax = plt.subplots(figsize=(11, 6))
unique_epochs = np.unique(epoch_number)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_epochs)))

for color, current_epoch in zip(colors, unique_epochs):
    mask = (epoch_number == current_epoch) & (np.abs(phase_days) < PLOT_WINDOW_D)
    if not np.any(mask):
        continue
    ax.errorbar(
        phase_days[mask],
        flat_lc.flux.value[mask],
        yerr=flat_lc.flux_err.value[mask],
        fmt="o",
        ms=4,
        color=color,
        ecolor="0.75",
        elinewidth=0.8,
        capsize=0,
        alpha=0.85,
        label=f"Orbit {current_epoch}",
    )

ax.set_title("Epoch-to-epoch transit overlay")
ax.set_xlabel("Time from mid-transit [days]")
ax.set_ylabel("Normalized flux")
ax.grid(alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, ncol=1)
plt.tight_layout()



## Joint Multi-Sector Transit Model

The next cells fit all downloaded TESS sectors jointly with a plain transit model.

- all sectors share the transit geometry (`period`, `t0`, `r`, `b`, `rho_star`, limb darkening)
- each sector gets its own flux offset and extra white-noise jitter term
- no GP is used in this version; the goal is to recover a physically sensible transit solution first


In [ ]:
try:
    import pymc as pm
    import pytensor.tensor as pt
    import exoplanet as xo
except Exception as exc:
    print(
        "Optional modeling dependencies could not be imported. "
        "This is usually either a missing package or an incompatible exoplanet / JAX combination."
    )
    print(f"Import failure: {type(exc).__name__}: {exc}")
else:
    RHO_SUN_CGS = 1.408
    MODEL_WINDOW_D = PLOT_WINDOW_D

    sector_labels = []
    x_parts = []
    y_parts = []
    yerr_parts = []
    sector_idx_parts = []

    for sector_number, raw_lc in enumerate(lc_collection, start=1):
        sector_label = raw_lc.meta.get("MISSION", f"sector_{sector_number}")
        sector_lc = raw_lc.remove_nans().normalize().remove_outliers(sigma=5.0)
        transit_mask = sector_lc.create_transit_mask(
            period=PERIOD_D,
            transit_time=T0_BTJD,
            duration=TRANSIT_DURATION_D,
        )
        sector_flat = sector_lc.flatten(window_length=901, mask=transit_mask)

        sector_phase = ((sector_flat.time.value - T0_BTJD + 0.5 * PERIOD_D) % PERIOD_D) - 0.5 * PERIOD_D
        window_mask = np.abs(sector_phase) < MODEL_WINDOW_D
        if not np.any(window_mask):
            continue

        flux_err = np.asarray(sector_flat.flux_err.value, dtype=np.float64)
        finite_err = flux_err[np.isfinite(flux_err) & (flux_err > 0)]
        err_fill = float(np.median(finite_err)) if finite_err.size else 5.0e-4
        flux_err = np.where(np.isfinite(flux_err) & (flux_err > 0), flux_err, err_fill)

        x_parts.append(np.ascontiguousarray(sector_flat.time.value[window_mask], dtype=np.float64))
        y_parts.append(np.ascontiguousarray(sector_flat.flux.value[window_mask] - 1.0, dtype=np.float64))
        yerr_parts.append(np.ascontiguousarray(flux_err[window_mask], dtype=np.float64))
        sector_idx_parts.append(np.full(np.count_nonzero(window_mask), len(sector_labels), dtype=np.int64))
        sector_labels.append(sector_label)

    x = np.concatenate(x_parts)
    y = np.concatenate(y_parts)
    yerr = np.concatenate(yerr_parts)
    sector_idx = np.concatenate(sector_idx_parts)

    sort_idx = np.argsort(x)
    x = x[sort_idx]
    y = y[sort_idx]
    yerr = yerr[sort_idx]
    sector_idx = sector_idx[sort_idx]

    n_sectors = len(sector_labels)
    sector_counts = np.bincount(sector_idx, minlength=n_sectors)
    sector_index_t = pt.as_tensor_variable(sector_idx)

    print(f"Fitting {n_sectors} sectors jointly")
    for label, count in zip(sector_labels, sector_counts):
        print(f"  {label}: {int(count)} cadences in the transit window")

    rho_star_guess = 0.3938 * RHO_SUN_CGS
    rho_star_sigma = 0.10 * RHO_SUN_CGS

    with pm.Model() as model:
        period = pm.Normal("period", mu=PERIOD_D, sigma=2.0e-4)
        t0 = pm.Normal("t0", mu=T0_BTJD, sigma=3.0e-3)
        r = pm.TruncatedNormal("r", mu=0.116, sigma=0.03, lower=0.01, upper=0.25, initval=0.116)
        b = pm.TruncatedNormal("b", mu=0.60, sigma=0.25, lower=0.0, upper=1.2, initval=0.60)
        rho_star = pm.TruncatedNormal(
            "rho_star",
            mu=rho_star_guess,
            sigma=rho_star_sigma,
            lower=0.05,
            upper=3.0,
            initval=rho_star_guess,
        )
        u = xo.distributions.QuadLimbDark("u")

        mean_flux = pm.Normal("mean_flux", mu=0.0, sigma=5.0e-4, shape=n_sectors)
        log_jitter = pm.Normal("log_jitter", mu=np.log(np.median(yerr)), sigma=2.0, shape=n_sectors)

        orbit = xo.orbits.KeplerianOrbit(period=period, t0=t0, b=b, rho_star=rho_star)
        light_curves = xo.LimbDarkLightCurve(u).get_light_curve(
            orbit=orbit,
            r=r,
            t=x,
            use_in_transit=False,
        )
        transit_model = pm.Deterministic("transit_model", pt.sum(light_curves, axis=-1))

        mu = pm.Deterministic("mean_model", transit_model + mean_flux[sector_index_t])
        jitter = pt.exp(log_jitter[sector_index_t])
        obs_sigma = pt.sqrt(yerr**2 + jitter**2)

        pm.Normal("obs", mu=mu, sigma=obs_sigma, observed=y)

        pm.Deterministic("transit_depth", r**2)
        pm.Deterministic("transit_depth_percent", 100.0 * r**2)
        pm.Deterministic("rho_star_solar", rho_star / RHO_SUN_CGS)
        pm.Deterministic("a_over_rstar", orbit.a / orbit.r_star)
        pm.Deterministic("inclination_deg", orbit.incl * 180.0 / np.pi)

    print("Joint multi-sector transit model built successfully.")


## MAP Optimization And Posterior Sampling

Run these after the joint multi-sector model cell succeeds.


In [ ]:
with model:
    map_soln = pm.find_MAP()

print({
    k: map_soln[k]
    for k in ["period", "t0", "r", "b", "rho_star"]
})

with model:
    idata = pm.sample(
        tune=1000,
        draws=1000,
        chains=2,
        cores=2,
        target_accept=0.9,
        init="adapt_diag",
    )


In [ ]:
import arviz as az

summary_vars = [
    "period",
    "t0",
    "r",
    "b",
    "rho_star",
    "rho_star_solar",
    "a_over_rstar",
    "inclination_deg",
    "transit_depth_percent",
    "mean_flux",
    "log_jitter",
]

az.summary(idata, var_names=summary_vars, round_to=6)


In [ ]:
trace_vars = ["period", "t0", "r", "b", "rho_star", "transit_depth_percent", "a_over_rstar", "inclination_deg"]
az.plot_trace(idata, var_names=trace_vars)
plt.tight_layout()


In [ ]:
posterior = idata.posterior

period_med = posterior["period"].median(dim=("chain", "draw")).item()
t0_med = posterior["t0"].median(dim=("chain", "draw")).item()
r_med = posterior["r"].median(dim=("chain", "draw")).item()
b_med = posterior["b"].median(dim=("chain", "draw")).item()
rho_star_med = posterior["rho_star"].median(dim=("chain", "draw")).item()
u_med = posterior["u"].median(dim=("chain", "draw")).values
mean_flux_med = posterior["mean_flux"].median(dim=("chain", "draw")).values
jitter_med = np.exp(posterior["log_jitter"].median(dim=("chain", "draw")).values)

orbit_med = xo.orbits.KeplerianOrbit(period=period_med, t0=t0_med, b=b_med, rho_star=rho_star_med)
light_curve_med = xo.LimbDarkLightCurve(u_med).get_light_curve(
    orbit=orbit_med,
    r=r_med,
    t=x,
    use_in_transit=False,
).eval()
transit_model_med = np.sum(light_curve_med, axis=-1)
mean_model_med = transit_model_med + mean_flux_med[sector_idx]
detrended_flux = y - mean_flux_med[sector_idx]
residual_flux = y - mean_model_med

fig, axes = plt.subplots(3, 1, figsize=(11, 11), sharex=True)

axes[0].plot(x, y, ".", color="0.4", alpha=0.22, ms=4, label="Data")
axes[0].plot(x, mean_model_med, color="tab:red", lw=2, label="Transit + sector offset")
axes[0].set_title("Joint multi-sector raw light curve fit")
axes[0].set_ylabel("Flux - 1")
axes[0].legend()

axes[1].plot(x, detrended_flux, ".", color="0.35", alpha=0.22, ms=4, label="Offset-corrected data")
axes[1].plot(x, transit_model_med, color="tab:blue", lw=2, label="Transit model")
axes[1].set_title("Offset-corrected light curve")
axes[1].set_ylabel("Detrended flux")
axes[1].legend()

axes[2].plot(x, residual_flux, ".", color="0.4", alpha=0.25, ms=4)
axes[2].axhline(0.0, color="tab:red", lw=1.5)
axes[2].set_title("Residuals")
axes[2].set_xlabel("Time [BTJD]")
axes[2].set_ylabel("Residual flux")

plt.tight_layout()

print("Median sector offsets:")
for label, offset, jitter in zip(sector_labels, mean_flux_med, jitter_med):
    print(f"  {label}: offset={offset:+.6e}, jitter={jitter:.6e}")


In [ ]:
phase_days = ((x - t0_med + 0.5 * period_med) % period_med) - 0.5 * period_med
sort_idx = np.argsort(phase_days)

bin_edges = np.linspace(-PLOT_WINDOW_D, PLOT_WINDOW_D, 81)
bin_index = np.digitize(phase_days, bin_edges)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
binned_phase = []
binned_flux = []
for idx in range(1, len(bin_edges)):
    mask = bin_index == idx
    if np.count_nonzero(mask) < 3:
        continue
    binned_phase.append(np.mean(phase_days[mask]))
    binned_flux.append(np.mean(detrended_flux[mask]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(phase_days, detrended_flux, ".", color="0.5", alpha=0.18, ms=4, label="All detrended cadences")
ax.plot(np.asarray(binned_phase), np.asarray(binned_flux), "o", color="tab:blue", ms=5, label="Binned data")
ax.plot(phase_days[sort_idx], transit_model_med[sort_idx], color="tab:red", lw=2, label="Transit model")
ax.set_xlim(-PLOT_WINDOW_D, PLOT_WINDOW_D)
ax.set_title("Phase-folded joint multi-sector transit")
ax.set_xlabel("Time from mid-transit [days]")
ax.set_ylabel("Detrended flux")
ax.legend()
plt.tight_layout()

def median_pm(samples):
    samples = np.asarray(samples).reshape(-1)
    q16, q50, q84 = np.percentile(samples, [16, 50, 84])
    return q50, q84 - q50, q50 - q16

r_stats = median_pm(posterior["r"].values)
b_stats = median_pm(posterior["b"].values)
rho_stats = median_pm(posterior["rho_star_solar"].values)
aor_stats = median_pm(posterior["a_over_rstar"].values)
depth_stats = median_pm(posterior["transit_depth_percent"].values)

print(f"Rp/R* = {r_stats[0]:.5f} +{r_stats[1]:.5f} -{r_stats[2]:.5f}")
print(f"b = {b_stats[0]:.4f} +{b_stats[1]:.4f} -{b_stats[2]:.4f}")
print(f"rho_star / rho_sun = {rho_stats[0]:.4f} +{rho_stats[1]:.4f} -{rho_stats[2]:.4f}")
print(f"a/R* = {aor_stats[0]:.4f} +{aor_stats[1]:.4f} -{aor_stats[2]:.4f}")
print(f"Transit depth (%) = {depth_stats[0]:.4f} +{depth_stats[1]:.4f} -{depth_stats[2]:.4f}")
print("Sectors fit:")
for label, count in zip(sector_labels, sector_counts):
    print(f"  {label}: {int(count)} cadences")
